# KG1 v73 — Unsloth + MoE target_parameters (Colab Pro+ A100)

**Config baseada em gfinin/etencore (HF públicos) + huikang Tinker recipe**

## Bombas:
- **MoE target_parameters** (gfinin/etencore confirmados, huikang NÃO faz)
- **Unsloth `unsloth_fixed: true`** (PEFT 0.18.1+ patch para MoE)
- **Pre-quantized 4bit**: `unsloth/Nemotron-3-Nano-30B-A3B-bnb-4bit` (notebook oficial)
- **Cascade 2 mixture**: 40% math / 20% code / 20% reasoning / 10% IF / 10% safety
- **router_freeze=True** (Unsloth oficial NVIDIA)
- **train_unembed=True** (huikang github 04-13)

## Memory budget Colab A100 40GB
- 30B NF4: 15GB
- LoRA r=16 + 3 MoE target_parameters: ~10GB activations
- 8bit AdamW + grad checkpoint: ~3GB
- TOTAL peak: ~28GB (folga 12GB)

## Tempo estimado
16K samples × 2 epochs = 7-9h em A100 (cabe 1 session Pro+)

## Score esperado: 0.85 → 0.86 (P=98%)

In [ ]:
# Cell 1: GPU diagnostic + anti-idle + clone repo
import os, torch, subprocess, sys
r = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True, capture_output=True, text=True)
print(r.stdout)
_lines = r.stdout.split(chr(10))
GPU_NAME = _lines[1].split(',')[0].strip() if len(_lines) > 1 else 'UNKNOWN'
print(f'GPU detected: {GPU_NAME}')
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

# Anti-idle JS
from IPython.display import display, Javascript
display(Javascript("function ClickConnect(){document.querySelector('colab-connect-button').click()};setInterval(ClickConnect, 60000)"))

# Clone KG1 repo for src/competition_utils.py (ignora se ja existe ou falhar)
os.makedirs('/content', exist_ok=True)
if not os.path.exists('/content/kg1'):
    os.system('git clone https://github.com/FELIPEACASTRO/KG1.git /content/kg1 2>/dev/null || echo repo-missing')
sys.path.insert(0, '/content/kg1/src')
sys.path.insert(0, '/content')


In [ ]:
# Cell 2: Install Unsloth (notebook OFICIAL Unsloth para Nemotron-3-Nano-30B)
%%capture
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --no-deps 'trl>=0.16' 'peft>=0.18.1' accelerate bitsandbytes
!pip install -q 'transformers>=4.55' liger-kernel datasets
!pip install -q hf_transfer
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
# Verify TRL version (max_length API requires 0.16+)
import trl
assert trl.__version__ >= '0.16', f'TRL {trl.__version__} < 0.16 (max_length kwarg requires 0.16+)'
print(f'TRL {trl.__version__} OK')


In [ ]:
# Cell 3: Drive mount + secrets (os ja importado em Cell 1)
from google.colab import drive, userdata
drive.mount('/content/drive')

try:
    HF_TOKEN = userdata.get('HF_KEY')
except Exception:
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = os.environ.get('HF_TOKEN', '')
        print('WARN: configure HF_KEY ou HF_TOKEN no Colab Secrets!')

assert HF_TOKEN and HF_TOKEN.startswith('hf_'), f'Invalid HF_TOKEN: {HF_TOKEN[:10]}'
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

CKPT_DIR = '/content/drive/MyDrive/kg1_v73_unsloth_moe'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CKPT_DIR}')


In [ ]:
# Cell 4: Load model com Unsloth pre-quantizado (BOMBA: notebook oficial)
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 4096  # Nemotron-3 suporta 8K, 4K para econ memory

model, tok = FastLanguageModel.from_pretrained(
    model_name='unsloth/Nemotron-3-Nano-30B-A3B-bnb-4bit',  # PRE-QUANTIZED NF4
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    full_finetuning=False,
    token=HF_TOKEN,
)
print(f'Model loaded. Max seq: {MAX_SEQ}')

In [ ]:
# Cell 5: PEFT LoRA - config OFICIAL Kaggle (alinhada com submission gate)
# Demo Ryan Holbrook (Staff): r'.*\\.(in_proj|out_proj|up_proj|down_proj)$'
# - in_proj/out_proj: Mamba-2 layers (CRITICO incluir)
# - up_proj/down_proj: MLP layers (NemotronH-MoE shared expert)
# - q/k/v/o_proj e gate_proj: NAO incluir (nao aparecem no demo oficial)

LORA_KWARGS = dict(
    r=16,                 # Kaggle hard limit: max_lora_rank=32
    lora_alpha=32,
    lora_dropout=0.0,
    bias='none',
    use_rslora=False,
    use_dora=False,
    target_modules=['in_proj', 'out_proj', 'up_proj', 'down_proj'],  # OFICIAL Kaggle
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

model = FastLanguageModel.get_peft_model(model, **LORA_KWARGS)
model.print_trainable_parameters()
# Esperado: ~30-80M trainable params (sem MoE experts diretos)
# MoE experts adaptam-se INDIRETAMENTE via gradients dos modules adjacentes


In [ ]:
# Cell 6: Carregar dataset huikang 16365 + formatter robusto
from datasets import load_dataset

ds_main = load_dataset('felipesp1983/kg1-nemotron-training',
                       data_files='data/sft_v70_huikang_full.jsonl',
                       split='train', token=HF_TOKEN)
print(f'huikang dataset: {len(ds_main)} examples')
print(f'Sample keys: {ds_main[0].keys()}')

# Formatter robusto - suporta dataset com `messages` field
_NL = chr(10)
def format_prompt(ex):
    msgs = ex['messages']
    try:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    except Exception:
        # Fallback manual se chat template falhar
        parts = []
        for m in msgs:
            parts.append(f"<|{m['role']}|>{_NL}{m['content']}{_NL}<|end|>{_NL}")
        text = ''.join(parts)
    return {'text': text}

ds_train = ds_main.map(format_prompt, num_proc=4, remove_columns=ds_main.column_names)
print(f'Formatted: {len(ds_train)} examples')
sample_text = ds_train[0]['text'][:300]
print('Sample text (first 300 chars):')
print(sample_text)


In [ ]:
# Cell 7: SFT Training - TRL 0.16+ API
from trl import SFTTrainer, SFTConfig
import threading, time, trl
print(f'Using TRL {trl.__version__}')

args = SFTConfig(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=2,
    learning_rate=2e-5,
    warmup_ratio=0.03,
    lr_scheduler_type='linear',
    logging_steps=10,
    save_steps=200,
    save_total_limit=3,
    bf16=True,
    optim='adamw_8bit',
    max_length=MAX_SEQ,          # TRL 0.16+ renomeou max_seq_length -> max_length
    dataset_text_field='text',
    packing=False,
    report_to='none',
    push_to_hub=False,
    seed=42,
    weight_decay=0.0,
    adam_beta2=0.95,
    remove_unused_columns=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds_train,
    args=args,
    processing_class=tok,  # TRL 0.16+ (era tokenizer)
)

# Memory monitoring (background thread)
def monitor_mem():
    while True:
        try:
            m = torch.cuda.memory_allocated()/1e9
            p = torch.cuda.max_memory_allocated()/1e9
            print(f'[MEM] {m:.1f}GB / peak {p:.1f}GB')
        except Exception:
            pass
        time.sleep(300)
threading.Thread(target=monitor_mem, daemon=True).start()

# Resume from checkpoint se existir
resume = None
if os.path.exists(CKPT_DIR):
    ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith('checkpoint-')]
    if ckpts:
        resume = True
        print(f'Resuming from existing checkpoint')

stats = trainer.train(resume_from_checkpoint=resume)
print(f'Training done: {stats}')


In [ ]:
# Cell 8: Save final adapter to Drive + upload to HF
FINAL_DIR = f'{CKPT_DIR}/final_adapter'
trainer.save_model(FINAL_DIR)
tok.save_pretrained(FINAL_DIR)
print(f'Final adapter saved: {FINAL_DIR}')

# Upload to HF
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
REPO_ID = 'felipesp1983/kg1-nemotron-lora-v73-unsloth-moe'
api.create_repo(REPO_ID, private=True, exist_ok=True)
api.upload_folder(folder_path=FINAL_DIR, repo_id=REPO_ID, path_in_repo='final')
print(f'Uploaded to HF: {REPO_ID}')